# Area charts

* It can be difficult to easily summarize GEE imageCollections and images over an area in an interactive environment
* geeViz provides two complementary approaches for area charting:
  1. **Map-based charting** (`canAreaChart` in `Map.addLayer`) — interactive charts inside the geeView map UI
  2. **Inline charting** (`geeViz.outputLib.charts`) — Plotly charts rendered directly in a notebook cell
* This notebook demonstrates both approaches side-by-side for each example



Copyright 2026 Ian Housman

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

[![github](https://img.shields.io/badge/-see%20sources-white?logo=github&labelColor=555)](https://github.com/gee-community/geeviz/blob/master/examples/areaChart_examples.ipynb) 
[![github](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gee-community/geeViz/blob/master/examples/areaChart_examples.ipynb)

In [1]:
#Import modules
try:
    import  geeViz.geeView as geeView
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geeViz'])
    import  geeViz.geeView as geeView

import geeViz.getImagesLib as gil
from geeViz.outputLib import charts as cl

[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:1112/ee-api (attached to existing detached proxy, tenant_header=X-geeViz-Creds)


geeViz: Earth Engine ready (project='geeviz-geo-agent', source=eeauth-proxy)


PyTables is not installed. No support for HDF output.


In [2]:
import pandas as pd
from IPython.display import display, HTML
ee = geeView.ee
Map = geeView.Map
Map.clearMap()
print("done")

done


* There are many different formats of data geeViz's area charting module can handle
* The most common is charting the percent or area in hectares or acres of thematic classes of an image collection
* If an imageCollection has `class_values`, `class_names`, `class_palette` properties for its images, all charts will automatically be populated with those names and colors

* First, we will look at these properties for a couple available images and image collections

In [3]:
lcms = ee.ImageCollection("projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11")
nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2021_REL/NLCD").select(['landcover'])
# Some releases store the *_class_* properties as comma-separated
# STRINGS rather than lists -- LCMS 2025-11 does. pd.DataFrame on three
# scalars raises "If using all scalar values, you must pass an index",
# so normalize before use. geeViz does the same thing internally (the
# autoViz path in geeView.py, and fsInsights.lcms_class_properties).
def listify(v):
    if isinstance(v, str):
        return [p.strip() for p in v.split(',') if p.strip()]
    return list(v or [])

# ...but NAMES can legitimately contain a comma, and then the split is
# wrong rather than merely inconvenient. LCMS 2025-11's Change class
# names include "Insect, Disease, or Drought Stress", so a comma split
# gives 18 parts for 16 classes -- every label after that entry would
# shift onto its neighbour's value. Which commas are separators is not
# recoverable from the string, so when the counts disagree this pads to
# the value count and says so instead of quietly mislabelling.
def align_names(names, values, bandName):
    if len(names) == len(values):
        return names
    print(f'  NOTE: {bandName} has {len(values)} class values but the '
          f'class_names string splits into {len(names)} parts, so at least '
          f'one name contains a comma. Names cannot be matched to values '
          f'here -- showing the raw split, padded.')
    return (names + [''] * len(values))[:len(values)]

def get_props_dataFrames(props,bandNames = None):
    props = {k:i for k,i in props.items() if k.find('_class_')>-1}

    if bandNames == None:
        bandNames = list(set([k.split('_class_')[0] for k in props.keys()]))
    out = {}
    for bn in bandNames:
        print(bn)
        values = listify(props[f'{bn}_class_values'])
        df = pd.DataFrame({'Values':values,
                           'Names':align_names(
                               listify(props[f'{bn}_class_names']),
                               values, bn),
                           'Colors':listify(props[f'{bn}_class_palette'])})
        display(df)
        out[bn]=df
    # return out
lcms_props = lcms.first().toDictionary().getInfo()
nlcd_props = nlcd.first().toDictionary().getInfo()


lcms_thematic_bandNames = lcms.select(['Change','Land_Cover','Land_Use']).first().bandNames().getInfo()

nlcd_landcover_bandNames = ['landcover']

get_props_dataFrames(lcms_props,lcms_thematic_bandNames)
get_props_dataFrames(nlcd_props,nlcd_landcover_bandNames)


# Shorten NLCD class names for later use
nlcd_class_names_key = f'{nlcd_landcover_bandNames[0]}_class_names'
nlcd_class_names_shortened = [nm.split(':')[0] for nm in listify(nlcd_props[nlcd_class_names_key])]
nlcd_props_shortened_names = nlcd_props
nlcd_props_shortened_names[nlcd_class_names_key] = nlcd_class_names_shortened

print('Shortened NLCD class names:')
get_props_dataFrames(nlcd_props_shortened_names,nlcd_landcover_bandNames)

Change
  NOTE: Change has 16 class values but the class_names string splits into 18 parts, so at least one name contains a comma. Names cannot be matched to values here -- showing the raw split, padded.


,Values,Names,Colors
0,1,Wind,ff09f3
1,2,Hurricane,541aff
2,3,Snow or Ice Transition,e4f5fd
3,4,Desiccation,cc982e
4,5,Inundation,0adaff
5,6,Prescribed Fire,a10018
6,7,Wildfire,d54309
7,8,Mechanical Land Transformation,fafa4b
8,9,Tree Removal,afde1c
9,10,Defoliation,ffc80d


Land_Cover


,Values,Names,Colors
0,1,Trees,004e2b
1,2,Tall Shrubs & Trees Mix (AK Only),009344
2,3,Shrubs & Trees Mix,61bb46
3,4,Grass/Forb/Herb & Trees Mix,acbb67
4,5,Barren & Trees Mix,8b8560
5,6,Tall Shrubs (AK Only),cafd4b
6,7,Shrubs,f89a1c
7,8,Grass/Forb/Herb & Shrubs Mix,8fa55f
8,9,Barren & Shrubs Mix,bebb8e
9,10,Grass/Forb/Herb,e5e98a


Land_Use


,Values,Names,Colors
0,1,Agriculture,fbff97
1,2,Developed,e6558b
2,3,Forest,004e2b
3,4,Other,9dbac5
4,5,Rangeland or Pasture,a6976a
5,6,Non-Processing Area Mask,1b1716


landcover


,Values,Names,Colors
0,11,"Open water: areas of open water, generally wit...",466b9f
1,12,Perennial ice/snow: areas characterized by a p...,d1def8
2,21,"Developed, open space: areas with a mixture of...",dec5c5
3,22,"Developed, low intensity: areas with a mixture...",d99282
4,23,"Developed, medium intensity: areas with a mixt...",eb0000
5,24,Developed high intensity: highly developed are...,ab0000
6,31,Barren land (rock/sand/clay): areas of bedrock...,b3ac9f
7,41,Deciduous forest: areas dominated by trees gen...,68ab5f
8,42,Evergreen forest: areas dominated by trees gen...,1c5f2c
9,43,Mixed forest: areas dominated by trees general...,b5c58f


Shortened NLCD class names:
landcover


,Values,Names,Colors
0,11,Open water,466b9f
1,12,Perennial ice/snow,d1def8
2,21,"Developed, open space",dec5c5
3,22,"Developed, low intensity",d99282
4,23,"Developed, medium intensity",eb0000
5,24,Developed high intensity,ab0000
6,31,Barren land (rock/sand/clay),b3ac9f
7,41,Deciduous forest,68ab5f
8,42,Evergreen forest,1c5f2c
9,43,Mixed forest,b5c58f


## Define a Study Area for Inline Charts

* The map-based charting uses the current map extent (or a selected feature) as the summary area
* For the inline `outputLib.charts` examples we need to define an explicit geometry
* This study area is reused throughout the notebook

In [4]:
# Study area: a rectangle in the western US (Colorado front range area)
study_area = ee.Geometry.Polygon(
    [[[-106.0, 39.5], [-105.0, 39.5], [-105.0, 40.5], [-106.0, 40.5], [-106.0, 39.5]]]
)
print('Study area defined')

Study area defined


---
## Basic Area Charting

* This example will show the most basic method for adding area chart layers

* By setting `"canAreaChart":True`, the layer will be added to area charting. Available properties (shown above) will be used to set up names and colors

* There are many options for area charting. Most of these options are provided as a dictionary using the `areaChartParams` key. This example shows two different `chartType`s 

* Using the `Map.turnOnAutoAreaCharting()` method will turn on autmatic area charting. This will use the map extent as the summary area.

* Additional methods are `Map.turnOnSelectionAreaCharting()` and `Map.turnOnUserDefinedAreaCharting()` for turning on different methods of providing areas to summarize. You can also change the method being used in the geeView UI under `Tools -> Area Tools`

In [5]:
# Map-based approach
Map.clearMap()


Map.addLayer(lcms.select(['Land_Cover']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'chartType':'stacked-line'}},'LCMS Land Cover')
Map.addLayer(nlcd.select(['landcover']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'chartType':'stacked-bar'}},'NLCD Land Cover')



Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMS Land Cover
Adding layer: NLCD Land Cover
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650299724&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-b26153e7ac8d.json


### Inline outputLib.charts equivalent

* `cl.summarize_and_chart` auto-detects thematic data and uses `frequencyHistogram`
* Returns both a `pandas.DataFrame` and a `plotly.graph_objects.Figure`
* Use `chart_type='stacked_bar'` for a stacked bar chart

In [6]:
# Inline: Stacked time series of LCMS Land Cover
result = cl.summarize_and_chart(
    lcms.select(['Land_Cover']),
    study_area,
    chart_type='stacked_line+markers',
    class_visible={'Non-Processing Area Mask':False},
    title='LCMS Land Cover'
)
df, fig = result['df'], result['chart']
fig.show()
display(df)

,Trees,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask,Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix
1985,56.41,21.09,0.01,6.71,0.10,1.43,4.21,0.80,0.51,0.27,3.90,4.43,0.14
1986,58.33,20.24,0.01,8.62,0.10,1.46,0.90,0.86,0.54,0.31,3.10,5.39,0.16
1987,58.78,20.37,0.01,8.69,0.06,1.45,0.08,0.88,0.54,0.34,2.91,5.73,0.16
1988,59.26,20.07,0.01,8.41,0.06,1.44,0.03,0.88,0.57,0.32,2.84,5.94,0.17
1989,59.78,19.61,0.01,8.31,0.05,1.41,0.01,0.85,0.57,0.34,2.89,6.01,0.16
1990,60.14,19.45,0.01,8.27,0.05,1.42,0.01,0.84,0.57,0.33,2.84,5.93,0.15
1991,60.08,19.47,0.01,8.41,0.05,1.43,0.01,0.85,0.55,0.33,2.86,5.82,0.14
1992,60.32,19.40,0.01,8.25,0.05,1.43,0.01,0.87,0.60,0.33,2.79,5.80,0.15
1993,60.65,18.89,0.01,8.24,0.05,1.45,0.00,0.83,0.59,0.32,2.72,6.08,0.16
1994,60.47,19.15,0.01,8.51,0.05,1.46,0.00,0.88,0.57,0.30,2.62,5.82,0.14


In [7]:
# Inline: Stacked bar chart of NLCD Land Cover
result = cl.summarize_and_chart(
    nlcd.select(['landcover']),
    study_area,
    title='NLCD Land Cover',
    chart_type='stacked_bar',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

,Open water: areas ... or soil.,Perennial ice/snow... cover.\r\n,"Developed, open sp...rposes.\r\n","Developed, low int... units.\r\n","Developed, medium ... units.\r\n",Developed high int... cover.\r\n,Barren land (rock/... cover.\r\n,Deciduous forest: ...change.\r\n,Evergreen forest: ...oliage.\r\n,Mixed forest: area... cover.\r\n,Shrub/scrub: areas...itions.\r\n,Grassland/herbaceo...razing.\r\n,Pasture/hay: areas...tation.\r\n,Cultivated crops: ...tilled.\r\n,Woody wetlands: ar... water.\r\n,Emergent herbaceou... water.\r\n
2021,1.61,0.99,3.93,4.53,3.49,1.07,1.43,1.18,43.72,0.11,17.07,13.77,1.12,2.84,1.71,1.44


---
## Line and Sankey Charts

* For thematic imageCollections, line and sankey charts are supported

* You can specify one or both within the `areaChartParams` dictionary. By default, line is chosen. (e.g. `"areaChartParams":{"sankey":True}`)

* Note that sankey charts work well at showing transitions between classes. Since the number of transition classes is the number of classes<sup>2</sup>, if there are many classes, sankey charting will be quite slow and may error out.

* For sankey charting, you can specify transition periods in the code using the `sankeyTransitionPeriods` key in `areaChartParams` (e.g. `"sankeyTransitionPeriods":[[1985,1987],[2000,2002],[2020,2022]]`), or leave it blank and geeViz will try to figure out a good set given the years of the provided imageCollection. Note that if you add imageCollections with different time extents, geeViz will take the intersection for the default years. You can add and change the periods in the geeView UI as well under `Tools -> Area Tools -> Area Tools Parameters` 

In [8]:
# Map-based approach
Map.clearMap()

Map.addLayer(lcms.select(['Change']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'sankey':True}},'LCMS Change')

Map.addLayer(lcms.select(['Land_Cover']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'sankey':True}},'LCMS Land Cover')

Map.addLayer(lcms.select(['Land_Use']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'sankey':True}},'LCMS Land Use')


Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMS Change
Adding layer: LCMS Land Cover
Adding layer: LCMS Land Use
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650305961&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-9f2e764f537d.json


### Inline outputLib.charts equivalent

* Set `chart_type='sankey'` and provide `transition_periods` as a list of `[start_year, end_year]` pairs
* `min_percentage` filters out small flows for readability

In [9]:
# Inline: Sankey diagram of LCMS Change transitions
result = cl.summarize_and_chart(
    lcms.select(['Change']),
    study_area,
    chart_type='sankey',
    transition_periods=[1985,1990,2000,2010,2024],
    title='LCMS Change Transitions',
    min_percentage=0.5,
)
sankey_df, sankey_fig, matrix_dict = result['df'], result['chart'], result['matrix']
display(sankey_fig)
display(sankey_df)
print('Transition matrices:')
for label, mdf in matrix_dict.items():
    print(f"\n{label}")
    display(mdf)

'<!DOCTYPE html>\n<html><head><meta charset="utf-8">\n<title>Sankey Diagram</title>\n<script src="https://unpkg.com/d3@7"></script>\n<script src="https://unpkg.com/d3-sankey@0"></script>\n<style>\n  * { margin: 0; padding: 0; box-sizing: border-box; }\n  body { background: #272822; font-family: Roboto, Arial, sans-serif; }\n  .plotly-graph-div { margin: 0 auto; }\n  #sankey { display: block; margin: 0 auto; }\n  #title {\n    text-align: center; padding: 16px 0 4px 0;\n    font-size: 16px; font-weight: 600; color: #f8f8f2;\n  }\n  #toolbar {\n    position: absolute; top: 8px; right: 16px;\n    display: flex; gap: 6px;\n  }\n  #toolbar button {\n    background: rgba(150,139,131,0.15); border: 1px solid rgba(150,139,131,0.3);\n    color: #f8f8f2; padding: 4px 12px; border-radius: 4px; cursor: pointer;\n    font-size: 12px;\n  }\n  #toolbar button:hover { background: rgba(150,139,131,0.3); }\n  .link:hover { stroke-opacity: 0.8 !important; }\n  .node-label {\n    fill: #f8f8f2; font-size:

,source,target,value,source_name,target_name,source_color,target_color,period
0,11,30,1.00,1985 Insect,1990 Other Loss,#f39268,#3d4551,1985 -> 1990
1,12,30,0.92,1985 Disease,1990 Other Loss,#c291d5,#3d4551,1985 -> 1990
2,13,29,0.96,1985 or Drought Stress,1990 or Drought Stress,#00a398,#00a398,1985 -> 1990
3,13,30,2.26,1985 or Drought Stress,1990 Other Loss,#00a398,#3d4551,1985 -> 1990
4,14,27,0.54,1985 Other Loss,1990 Insect,#3d4551,#f39268,1985 -> 1990
5,14,29,1.14,1985 Other Loss,1990 or Drought Stress,#3d4551,#00a398,1985 -> 1990
6,14,30,88.60,1985 Other Loss,1990 Other Loss,#3d4551,#3d4551,1985 -> 1990
7,15,30,4.03,1985 Vegetation Successional Growth,1990 Other Loss,#1b1716,#3d4551,1985 -> 1990
8,29,46,1.95,1990 or Drought Stress,2000 Other Loss,#00a398,#3d4551,1990 -> 2000
9,30,43,4.35,1990 Other Loss,2000 Insect,#3d4551,#f39268,1990 -> 2000


Transition matrices:

1985 → 1990


,Wind,Hurricane,Snow or Ice Transition,Desiccation,Inundation,Prescribed Fire,Wildfire,Mechanical Land Transformation,Tree Removal,Defoliation,Southern Pine Beetle,Insect,Disease,or Drought Stress,Other Loss,Vegetation Successional Growth
Wind,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Hurricane,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Snow or Ice Transition,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Desiccation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Inundation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Prescribed Fire,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Wildfire,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Mechanical Land Transformation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.07,0.00
Tree Removal,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00
Defoliation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.00



1990 → 2000


,Wind,Hurricane,Snow or Ice Transition,Desiccation,Inundation,Prescribed Fire,Wildfire,Mechanical Land Transformation,Tree Removal,Defoliation,Southern Pine Beetle,Insect,Disease,or Drought Stress,Other Loss,Vegetation Successional Growth
Wind,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Hurricane,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Snow or Ice Transition,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.01,0.0
Desiccation,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.02,0.0
Inundation,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.02,0.0
Prescribed Fire,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Wildfire,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Mechanical Land Transformation,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.01,0.0
Tree Removal,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.01,0.01,0.0
Defoliation,0,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0



2000 → 2010


,Wind,Hurricane,Snow or Ice Transition,Desiccation,Inundation,Prescribed Fire,Wildfire,Mechanical Land Transformation,Tree Removal,Defoliation,Southern Pine Beetle,Insect,Disease,or Drought Stress,Other Loss,Vegetation Successional Growth
Wind,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Hurricane,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Snow or Ice Transition,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.01,0.0
Desiccation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.04,0.0
Inundation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.02,0.0
Prescribed Fire,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0
Wildfire,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.02,0.23,0.0
Mechanical Land Transformation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.05,0.0
Tree Removal,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.01,0.0
Defoliation,0,0,0.0,0.00,0.00,0,0.0,0.00,0.00,0,0,0.00,0.00,0.00,0.00,0.0



2010 → 2024


,Wind,Hurricane,Snow or Ice Transition,Desiccation,Inundation,Prescribed Fire,Wildfire,Mechanical Land Transformation,Tree Removal,Defoliation,Southern Pine Beetle,Insect,Disease,or Drought Stress,Other Loss,Vegetation Successional Growth
Wind,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.00,0.0
Hurricane,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.00,0.0
Snow or Ice Transition,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.01,0.0
Desiccation,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.02,0.0
Inundation,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.03,0.0
Prescribed Fire,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.00,0.0
Wildfire,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.00,0.0
Mechanical Land Transformation,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.01,0.0
Tree Removal,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.03,0.05,0.0
Defoliation,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0.00,0.00,0.00,0.0


In [10]:
# Inline: Sankey diagram of LCMS Land Cover transitions
result = cl.summarize_and_chart(
    lcms.select(['Land_Cover']),
    study_area,
    chart_type='sankey',
    transition_periods=[1985, 1990, 2000, 2005, 2018, 2023],
    title='LCMS Land Cover Transitions',
    min_percentage=0.5,
)
sankey_df, sankey_fig, matrix_dict = result['df'], result['chart'], result['matrix']
display(HTML(sankey_fig))
print('Source-Target-Value table:')
display(sankey_df)
print('Transition matrices:')
for label, mdf in matrix_dict.items():
    print(f"\n{label}")
    display(mdf)

Source-Target-Value table:


,source,target,value,source_name,target_name,source_color,target_color,period
0,9,15,1.40,1985 Grass/Forb/Herb,1990 Trees,#e5e98a,#004e2b,1985 -> 1990
1,9,22,1.23,1985 Grass/Forb/Herb,1990 Grass/Forb/Herb & Shrubs Mix,#e5e98a,#8fa55f,1985 -> 1990
2,9,24,16.74,1985 Grass/Forb/Herb,1990 Grass/Forb/Herb,#e5e98a,#e5e98a,1985 -> 1990
3,9,26,1.35,1985 Grass/Forb/Herb,1990 Barren or Impervious,#e5e98a,#893f54,1985 -> 1990
4,11,15,0.72,1985 Barren or Impervious,1990 Trees,#893f54,#004e2b,1985 -> 1990
...,...,...,...,...,...,...,...,...
62,60,86,0.71,2018 Trees,2023 Barren or Impervious,#004e2b,#893f54,2018 -> 2023
63,62,77,0.51,2018 Shrubs & Trees Mix,2023 Shrubs & Trees Mix,#61bb46,#61bb46,2018 -> 2023
64,66,81,1.74,2018 Shrubs,2023 Shrubs,#f89a1c,#f89a1c,2018 -> 2023
65,67,82,4.05,2018 Grass/Forb/Herb & Shrubs Mix,2023 Grass/Forb/Herb & Shrubs Mix,#8fa55f,#8fa55f,2018 -> 2023


Transition matrices:

1985 → 1990


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,55.21,0,0.18,0.08,0.01,0,0.17,0.15,0.01,0.47,0.0,0.10,0.00,0.00,0.00
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00
Shrubs & Trees Mix,0.23,0,0.51,0.00,0.00,0,0.03,0.01,0.00,0.02,0.0,0.00,0.00,0.00,0.00
Grass/Forb/Herb & Trees Mix,0.16,0,0.02,0.21,0.00,0,0.02,0.04,0.00,0.05,0.0,0.01,0.00,0.00,0.00
Barren & Trees Mix,0.02,0,0.00,0.01,0.17,0,0.01,0.01,0.01,0.01,0.0,0.03,0.00,0.00,0.00
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00
Shrubs,0.40,0,0.04,0.03,0.01,0,2.16,0.80,0.00,0.43,0.0,0.03,0.00,0.00,0.00
Grass/Forb/Herb & Shrubs Mix,0.27,0,0.03,0.04,0.01,0,0.14,3.40,0.01,0.45,0.0,0.07,0.00,0.00,0.00
Barren & Shrubs Mix,0.02,0,0.00,0.01,0.01,0,0.01,0.01,0.07,0.01,0.0,0.01,0.00,0.00,0.00
Grass/Forb/Herb,1.40,0,0.05,0.08,0.02,0,0.20,1.23,0.01,16.74,0.0,1.35,0.00,0.02,0.00



1990 → 2000


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,58.09,0,0.25,0.16,0.02,0,0.22,0.35,0.02,0.81,0.00,0.22,0.00,0.01,0.0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Shrubs & Trees Mix,0.23,0,0.50,0.01,0.00,0,0.03,0.03,0.00,0.04,0.00,0.00,0.00,0.00,0.0
Grass/Forb/Herb & Trees Mix,0.16,0,0.01,0.24,0.01,0,0.01,0.05,0.01,0.07,0.00,0.01,0.00,0.00,0.0
Barren & Trees Mix,0.02,0,0.00,0.01,0.21,0,0.01,0.02,0.01,0.03,0.00,0.02,0.00,0.00,0.0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Shrubs,0.29,0,0.04,0.02,0.01,0,1.53,0.59,0.01,0.31,0.00,0.03,0.00,0.00,0.0
Grass/Forb/Herb & Shrubs Mix,0.26,0,0.03,0.05,0.01,0,0.11,4.69,0.01,0.60,0.00,0.18,0.00,0.00,0.0
Barren & Shrubs Mix,0.01,0,0.00,0.01,0.01,0,0.01,0.01,0.09,0.01,0.00,0.01,0.00,0.00,0.0
Grass/Forb/Herb,1.00,0,0.04,0.07,0.02,0,0.15,1.12,0.01,15.97,0.00,0.99,0.00,0.07,0.0


... (2 outputs left out)


2005 → 2018


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,56.19,0,0.49,0.45,0.02,0,0.93,0.44,0.02,1.37,0.00,0.26,0.00,0.01,0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs & Trees Mix,0.37,0,0.46,0.01,0.00,0,0.04,0.02,0.00,0.05,0.00,0.00,0.00,0.00,0
Grass/Forb/Herb & Trees Mix,0.17,0,0.01,0.27,0.01,0,0.01,0.06,0.01,0.09,0.00,0.02,0.00,0.00,0
Barren & Trees Mix,0.02,0,0.00,0.02,0.16,0,0.01,0.03,0.02,0.05,0.00,0.03,0.00,0.00,0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs,0.25,0,0.04,0.04,0.03,0,1.30,0.19,0.02,0.25,0.00,0.06,0.00,0.00,0
Grass/Forb/Herb & Shrubs Mix,0.50,0,0.05,0.08,0.03,0,0.24,4.77,0.01,1.23,0.00,0.24,0.00,0.01,0
Barren & Shrubs Mix,0.01,0,0.00,0.01,0.02,0,0.00,0.02,0.08,0.02,0.00,0.02,0.00,0.00,0
Grass/Forb/Herb,1.13,0,0.07,0.12,0.07,0,0.18,0.97,0.03,14.33,0.00,0.76,0.00,0.07,0



2018 → 2023


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,54.70,0,0.47,0.59,0.02,0,0.90,0.49,0.02,1.17,0.00,0.71,0.00,0.01,0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs & Trees Mix,0.42,0,0.51,0.02,0.00,0,0.10,0.02,0.00,0.05,0.00,0.00,0.00,0.00,0
Grass/Forb/Herb & Trees Mix,0.25,0,0.02,0.45,0.02,0,0.04,0.10,0.01,0.12,0.00,0.03,0.00,0.00,0
Barren & Trees Mix,0.01,0,0.00,0.02,0.20,0,0.01,0.03,0.02,0.05,0.00,0.03,0.00,0.00,0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs,0.27,0,0.09,0.07,0.04,0,1.74,0.20,0.02,0.27,0.00,0.07,0.00,0.00,0
Grass/Forb/Herb & Shrubs Mix,0.36,0,0.03,0.13,0.04,0,0.48,4.05,0.01,1.36,0.00,0.17,0.00,0.00,0
Barren & Shrubs Mix,0.01,0,0.00,0.02,0.03,0,0.00,0.02,0.07,0.03,0.00,0.02,0.00,0.00,0
Grass/Forb/Herb,0.95,0,0.08,0.19,0.09,0,0.32,0.92,0.03,14.89,0.00,0.45,0.00,0.02,0


In [11]:
# Inline: Sankey diagram of LCMS Land Use transitions
result = cl.summarize_and_chart(
    lcms.select(['Land_Use']),
    study_area,
    chart_type='sankey',
    transition_periods=[1985, 1990, 2005, 2010, 2018, 2023],
    title='LCMS Land Use Transitions',
    min_percentage=0.5,
)
sankey_df, sankey_fig, matrix_dict = result['df'], result['chart'], result['matrix']
display(HTML(sankey_fig))
print('Source-Target-Value table:')
display(sankey_df)
print('Transition matrices:')
for label, mdf in matrix_dict.items():
    print(f"\n{label}")
    display(mdf)

Source-Target-Value table:


,source,target,value,source_name,target_name,source_color,target_color,period
0,0,6,5.87,1985 Agriculture,1990 Agriculture,#fbff97,#fbff97,1985 -> 1990
1,1,7,6.67,1985 Developed,1990 Developed,#e6558b,#e6558b,1985 -> 1990
2,2,8,54.78,1985 Forest,1990 Forest,#004e2b,#004e2b,1985 -> 1990
3,2,10,0.84,1985 Forest,1990 Rangeland or Pasture,#004e2b,#a6976a,1985 -> 1990
4,3,8,0.70,1985 Other,1990 Forest,#9dbac5,#004e2b,1985 -> 1990
5,3,9,4.08,1985 Other,1990 Other,#9dbac5,#9dbac5,1985 -> 1990
6,3,10,0.50,1985 Other,1990 Rangeland or Pasture,#9dbac5,#a6976a,1985 -> 1990
7,4,8,1.82,1985 Rangeland or Pasture,1990 Forest,#a6976a,#004e2b,1985 -> 1990
8,4,10,19.46,1985 Rangeland or Pasture,1990 Rangeland or Pasture,#a6976a,#a6976a,1985 -> 1990
9,5,8,1.65,1985 Non-Processing Area Mask,1990 Forest,#1b1716,#004e2b,1985 -> 1990


Transition matrices:

1985 → 1990


,Agriculture,Developed,Forest,Other,Rangeland or Pasture,Non-Processing Area Mask
Agriculture,5.87,0.02,0.00,0.01,0.24,0.00
Developed,0.00,6.67,0.00,0.01,0.20,0.00
Forest,0.00,0.05,54.78,0.04,0.84,0.00
Other,0.01,0.01,0.70,4.08,0.50,0.00
Rangeland or Pasture,0.05,0.30,1.82,0.14,19.46,0.00
Non-Processing Area Mask,0.00,0.03,1.65,1.25,1.28,0.01



1990 → 2005


,Agriculture,Developed,Forest,Other,Rangeland or Pasture,Non-Processing Area Mask
Agriculture,4.72,0.51,0.01,0.03,0.66,0.0
Developed,0.00,6.82,0.00,0.01,0.23,0.0
Forest,0.00,0.24,57.10,0.04,1.57,0.0
Other,0.01,0.01,0.19,4.91,0.39,0.0
Rangeland or Pasture,0.08,1.73,1.68,0.19,18.85,0.0
Non-Processing Area Mask,0.00,0.00,0.00,0.01,0.00,0.0


... (2 outputs left out)


2010 → 2018


,Agriculture,Developed,Forest,Other,Rangeland or Pasture,Non-Processing Area Mask
Agriculture,3.90,0.12,0.01,0.01,0.32,0
Developed,0.00,9.36,0.00,0.01,0.37,0
Forest,0.00,0.07,56.42,0.08,1.95,0
Other,0.00,0.01,0.08,4.80,0.23,0
Rangeland or Pasture,0.06,0.52,1.51,0.40,19.77,0
Non-Processing Area Mask,0.00,0.00,0.00,0.00,0.00,0



2018 → 2023


,Agriculture,Developed,Forest,Other,Rangeland or Pasture,Non-Processing Area Mask
Agriculture,3.64,0.13,0.01,0.00,0.19,0
Developed,0.00,9.75,0.01,0.01,0.31,0
Forest,0.00,0.03,56.00,0.08,1.90,0
Other,0.00,0.01,0.10,4.80,0.38,0
Rangeland or Pasture,0.03,0.45,1.96,0.36,19.84,0
Non-Processing Area Mask,0.00,0.00,0.00,0.00,0.00,0


---
*  Adding a layer to line and sankey charting is done as follows

* Any time you specify `'line':True,'sankey':True`, both line and sankey charts will be created 

* This can slow things down a lot if many layers are visible. Also, moving the map, pausing for a second or two, and then moving the map again, many times, can create a long queue and delay charting.

In [12]:
# Map-based approach: both line and sankey
Map.clearMap()

Map.addLayer(lcms.select(['Change']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'line':True,'sankey':True}},'LCMS Change')

Map.addLayer(lcms.select(['Land_Cover']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'line':True,'sankey':True}},'LCMS Land Cover')

Map.addLayer(lcms.select(['Land_Use']),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'line':True,'sankey':True}},'LCMS Land Use')


Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMS Change
Adding layer: LCMS Land Cover
Adding layer: LCMS Land Use
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650308503&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-773a1df7492a.json


### Inline outputLib.charts equivalent: both line and sankey

* With `outputLib.charts`, you simply make two separate calls — one for the time series and one for the sankey

In [13]:
# Inline: both time series and sankey for LCMS Land Cover

# Time series
result = cl.summarize_and_chart(
    lcms.select(['Land_Cover']),
    study_area,
    title='LCMS Land Cover (time series)',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

# Sankey
result = cl.summarize_and_chart(
    lcms.select(['Land_Cover']),
    study_area,
    chart_type='sankey',
    transition_periods=[1985, 1990, 2005, 2010, 2018, 2023],
    title='LCMS Land Cover (sankey)',
    min_percentage=0.5,
)
sankey_df, sankey_fig, matrix_dict = result['df'], result['chart'], result['matrix']
display(HTML(sankey_fig))
print('Transition matrices:')
for label, mdf in matrix_dict.items():
    print(f"\n{label}")
    display(mdf)

Transition matrices:

1985 → 1990


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,55.21,0,0.18,0.08,0.01,0,0.17,0.15,0.01,0.47,0.0,0.10,0.00,0.00,0.00
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00
Shrubs & Trees Mix,0.23,0,0.51,0.00,0.00,0,0.03,0.01,0.00,0.02,0.0,0.00,0.00,0.00,0.00
Grass/Forb/Herb & Trees Mix,0.16,0,0.02,0.21,0.00,0,0.02,0.04,0.00,0.05,0.0,0.01,0.00,0.00,0.00
Barren & Trees Mix,0.02,0,0.00,0.01,0.17,0,0.01,0.01,0.01,0.01,0.0,0.03,0.00,0.00,0.00
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00
Shrubs,0.40,0,0.04,0.03,0.01,0,2.16,0.80,0.00,0.43,0.0,0.03,0.00,0.00,0.00
Grass/Forb/Herb & Shrubs Mix,0.27,0,0.03,0.04,0.01,0,0.14,3.40,0.01,0.45,0.0,0.07,0.00,0.00,0.00
Barren & Shrubs Mix,0.02,0,0.00,0.01,0.01,0,0.01,0.01,0.07,0.01,0.0,0.01,0.00,0.00,0.00
Grass/Forb/Herb,1.40,0,0.05,0.08,0.02,0,0.20,1.23,0.01,16.74,0.0,1.35,0.00,0.02,0.00



1990 → 2005


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,57.51,0,0.33,0.19,0.02,0,0.32,0.56,0.02,0.97,0.00,0.21,0.00,0.01,0.0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Shrubs & Trees Mix,0.26,0,0.46,0.01,0.00,0,0.03,0.03,0.00,0.04,0.00,0.00,0.00,0.00,0.0
Grass/Forb/Herb & Trees Mix,0.17,0,0.02,0.22,0.01,0,0.01,0.06,0.01,0.07,0.00,0.01,0.00,0.00,0.0
Barren & Trees Mix,0.02,0,0.00,0.01,0.19,0,0.01,0.03,0.01,0.04,0.00,0.01,0.00,0.00,0.0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Shrubs,0.34,0,0.05,0.02,0.01,0,1.43,0.61,0.01,0.34,0.00,0.04,0.00,0.00,0.0
Grass/Forb/Herb & Shrubs Mix,0.34,0,0.04,0.06,0.01,0,0.13,4.43,0.01,0.69,0.00,0.23,0.00,0.00,0.0
Barren & Shrubs Mix,0.02,0,0.00,0.01,0.01,0,0.01,0.01,0.08,0.01,0.00,0.01,0.00,0.00,0.0
Grass/Forb/Herb,1.15,0,0.05,0.08,0.03,0,0.18,1.31,0.02,15.16,0.00,1.39,0.00,0.09,0.0



2005 → 2010


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,58.26,0,0.27,0.17,0.01,0,0.44,0.21,0.01,0.72,0.00,0.10,0.00,0.01,0.0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Shrubs & Trees Mix,0.18,0,0.69,0.01,0.00,0,0.02,0.02,0.00,0.03,0.00,0.00,0.00,0.00,0.0
Grass/Forb/Herb & Trees Mix,0.12,0,0.01,0.39,0.01,0,0.01,0.04,0.01,0.06,0.00,0.01,0.00,0.00,0.0
Barren & Trees Mix,0.01,0,0.00,0.01,0.24,0,0.01,0.02,0.01,0.03,0.00,0.01,0.00,0.00,0.0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Shrubs,0.13,0,0.02,0.02,0.01,0,1.70,0.12,0.01,0.14,0.00,0.02,0.00,0.00,0.0
Grass/Forb/Herb & Shrubs Mix,0.21,0,0.03,0.04,0.01,0,0.11,5.99,0.01,0.66,0.00,0.10,0.00,0.00,0.0
Barren & Shrubs Mix,0.01,0,0.00,0.01,0.01,0,0.00,0.01,0.13,0.01,0.00,0.01,0.00,0.00,0.0
Grass/Forb/Herb,0.62,0,0.05,0.05,0.02,0,0.10,0.64,0.01,15.83,0.00,0.36,0.00,0.05,0.0



2010 → 2018


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,56.61,0,0.39,0.39,0.02,0,0.63,0.35,0.02,1.09,0.00,0.28,0.00,0.01,0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs & Trees Mix,0.38,0,0.55,0.01,0.00,0,0.05,0.02,0.00,0.06,0.00,0.00,0.00,0.00,0
Grass/Forb/Herb & Trees Mix,0.17,0,0.01,0.32,0.01,0,0.02,0.06,0.01,0.08,0.00,0.02,0.00,0.00,0
Barren & Trees Mix,0.02,0,0.00,0.01,0.17,0,0.01,0.02,0.02,0.04,0.00,0.04,0.00,0.00,0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs,0.24,0,0.05,0.05,0.03,0,1.56,0.17,0.02,0.25,0.00,0.06,0.00,0.00,0
Grass/Forb/Herb & Shrubs Mix,0.37,0,0.04,0.08,0.03,0,0.25,5.01,0.01,1.14,0.00,0.21,0.00,0.00,0
Barren & Shrubs Mix,0.01,0,0.00,0.01,0.02,0,0.00,0.02,0.08,0.02,0.00,0.02,0.00,0.00,0
Grass/Forb/Herb,0.95,0,0.07,0.13,0.07,0,0.22,0.88,0.03,14.82,0.00,0.59,0.00,0.03,0



2018 → 2023


,Trees,Tall Shrubs & Tree...(AK Only),Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Tall Shrubs (AK Only),Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask
Trees,54.70,0,0.47,0.59,0.02,0,0.90,0.49,0.02,1.17,0.00,0.71,0.00,0.01,0
Tall Shrubs & Tree...(AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs & Trees Mix,0.42,0,0.51,0.02,0.00,0,0.10,0.02,0.00,0.05,0.00,0.00,0.00,0.00,0
Grass/Forb/Herb & Trees Mix,0.25,0,0.02,0.45,0.02,0,0.04,0.10,0.01,0.12,0.00,0.03,0.00,0.00,0
Barren & Trees Mix,0.01,0,0.00,0.02,0.20,0,0.01,0.03,0.02,0.05,0.00,0.03,0.00,0.00,0
Tall Shrubs (AK Only),0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
Shrubs,0.27,0,0.09,0.07,0.04,0,1.74,0.20,0.02,0.27,0.00,0.07,0.00,0.00,0
Grass/Forb/Herb & Shrubs Mix,0.36,0,0.03,0.13,0.04,0,0.48,4.05,0.01,1.36,0.00,0.17,0.00,0.00,0
Barren & Shrubs Mix,0.01,0,0.00,0.02,0.03,0,0.00,0.02,0.07,0.03,0.00,0.02,0.00,0.00,0
Grass/Forb/Herb,0.95,0,0.08,0.19,0.09,0,0.32,0.92,0.03,14.89,0.00,0.45,0.00,0.02,0


---
## Ways of adding area charting layers
* There are two methods for adding area chart layers. The first is shown above, where when adding a layer using `Map.addLayer`, specify `"canAreaChart":True`. With this method, if a layer is visible, it will be included in area charting.
* There are instances where you would like to summarize layers, but may not want them on the map or it is impossible to visualize all the thematic bands you'd like to chart. In this instance, you can use the `Map.addAreaChartLayer` method.
* If you use the `Map.addAreaChartLayer` method, you will need to use the `Map.populateAreaChartLayerSelect()` method to instantiate a selection menu for choosing which area chart layers should be charted. 
* In this example, we will summarize all thematic classes in LCMS in a single graph. This cannot be displayed on a map, but is an interesting way to look at the summarized data in charts

* Note that the dictionary of parameters is more or less the same as what you would put in the `"areaChartParams"` if you were to use the `Map.addLayer` method. 

* Note that while multi thematic band image collections can be charted in a single line chart, sankey charts can only support one band per chart. If a multi thematic band image collection is given with `"sankey":True` (as is the case in this example), separate sankey charts will be created for each band.

In [14]:
# Map-based approach
Map.clearMap()

Map.addLayer(lcms.select(['Change_Raw_Probability.*']),{'reducer':ee.Reducer.stdDev(),'min':0,'max':10},'LCMS Change Prob')

Map.addAreaChartLayer(lcms,{'line':True},'LCMS All Thematic Classes Line',False)

Map.addAreaChartLayer(lcms,{'sankey':True},'LCMS All Thematic Classes Sankey',True)

Map.populateAreaChartLayerSelect()
Map.turnOnAutoAreaCharting()

Map.view()

Adding layer: LCMS Change Prob
Adding area chart layer: LCMS All Thematic Classes Line
Adding area chart layer: LCMS All Thematic Classes Sankey
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650312308&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-69bce49fd08b.json


### Inline outputLib.charts equivalent

* `outputLib.charts` can chart all thematic bands at once — just select multiple bands
* Column names are automatically prefixed with the band name when multiple bands are present

In [15]:
# Inline: All LCMS thematic bands in one chart
result = cl.summarize_and_chart(
    lcms.select(['Change', 'Land_Cover', 'Land_Use']),
    study_area,
    title='LCMS All Thematic Classes',
    chart_type='line+markers',
    line_width=1,
    width=800,
    height=500,
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df.head())

,Change----Insect,Change----Disease,Change----or Drought Stress,Change----Other Loss,Change----Vegetation Successional Growth,Change----Mechanical Land Transformation,Land_Cover----Trees,Land_Cover----Grass/Forb/Herb,Land_Cover----Barren & Grass/Forb/Herb Mix,Land_Cover----Barren or Impervious,...,Land_Use----Other,Land_Use----Rangeland or Pasture,Land_Use----Non-Processing Area Mask,Change----Snow or Ice Transition,Change----Desiccation,Change----Inundation,Change----Tree Removal,Change----Wildfire,Change----Defoliation,Change----Wind
1985,1.10,0.97,3.26,90.38,4.22,0.07,56.41,21.09,0.01,6.71,...,5.29,21.78,4.22,0.00,0.00,0.00,0.00,0.00,0.0,0.0
1986,1.08,0.27,3.30,94.32,0.90,0.02,58.33,20.24,0.01,8.62,...,5.95,22.58,0.90,0.03,0.03,0.03,0.01,0.00,0.0,0.0
1987,0.89,0.19,2.88,95.80,0.08,0.01,58.78,20.37,0.01,8.69,...,5.88,22.91,0.08,0.05,0.04,0.02,0.03,0.00,0.0,0.0
1988,0.57,0.04,3.11,96.17,0.03,0.01,59.26,20.07,0.01,8.41,...,5.67,22.83,0.03,0.01,0.03,0.02,0.01,0.00,0.0,0.0
1989,0.57,0.04,2.61,96.62,0.01,0.01,59.78,19.61,0.01,8.31,...,5.56,22.73,0.01,0.01,0.05,0.02,0.02,0.03,0.0,0.0


---
## Charting Non-Thematic Data

* You can chart continuous data as well. By default, a `ee.Reducer.mean()` will be used. You can use any reducer that returns a single value per image-band (e.g. `ee.Reducer.min()`, `ee.Reducer.max()`, `ee.Reducer.stdDev()`, `ee.Reducer.mode()`, and not `ee.Reducer.percentile([0,50,100])`). 

* You can specify this using `"areaChartParams":{"reducer":ee.Reducer.mean()}`

* Optionally, you can provide a color palette to be used. Each band will be assigned to a color in the order given

* Notice in the example, the reducer for what is shown on the map is different from the zonal summary reducer. In this example, on the map the standard deviation of the probability is shown to highlight likely change, while the average over the area is shown in the chart since that is a more appropriate representation of probability over an area. 

In [16]:
# Map-based approach
Map.clearMap()

Map.addLayer(lcms.select(['Change_Raw_Probability.*']),
             {'reducer':ee.Reducer.stdDev(),'min':0,'max':10,'canAreaChart':True,'areaChartParams':{'palette':'f39268,d54309,00a398'}},'LCMS Change Prob')


Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMS Change Prob
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650315009&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-a5496280ce0b.json


### Inline outputLib.charts equivalent

* `outputLib.charts` auto-detects continuous data (no class properties) and defaults to `ee.Reducer.mean()`
* For non-thematic data, `chart_type='line+markers` gives a multi-line chart

In [17]:
# Inline: Continuous data — LCMS change probability bands
result = cl.summarize_and_chart(
    lcms.select(['Change_Raw_Probability_Slow_Loss',
                 'Change_Raw_Probability_Fast_Loss',
                 'Change_Raw_Probability_Gain']),
    study_area,
    title='LCMS Change Probabilities (Mean)',
    chart_type = 'line+markers',
    marker_size=5,
    line_width=2,
    palette=['f39268', 'd54309', '00a398'],
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

,Change_Raw_Probability_Slow_Loss,Change_Raw_Probability_Fast_Loss,Change_Raw_Probability_Gain
1985,1.022331,1.923802,7.881063
1986,0.883890,1.068220,7.764327
1987,0.849171,0.985674,7.537322
1988,0.684917,0.506159,7.598007
1989,0.683735,0.508912,7.282339
1990,0.781505,0.499000,7.037582
1991,1.258071,0.587623,6.336037
1992,1.094287,0.447604,6.054147
1993,1.127077,0.433620,5.835327
1994,1.467598,0.477430,5.573199


---
## Charting Images

* You can also chart images
* It will behave in a similar fashion to imageCollections, but will show a bar chart
* Placing names for bar charts can be challenging if the names are very long. geeViz will automatically change the bar chart to be a horizontal bar chart if long names are detected. This still does not ensure the bar charts are readable (as is the case in this example). Shortening class names is the easiest method to address this issue (shown by splitting the full NLCD landcover name with `:` and take the first part earlier in this notebook).
* If using `"autoViz":True`, be sure to copy the `_class_` properties back in

In [18]:
# Map-based approach
Map.clearMap()


Map.addLayer(lcms.select(['Land_Cover']),{'autoViz':True,'canAreaChart':True},'LCMS Land Cover')
Map.addLayer(lcms.select(['Land_Cover']).mode().set(lcms.first().toDictionary()),{'autoViz':True,'canAreaChart':True},'LCMS Land Cover Mode')

Map.addLayer(nlcd.select(['landcover']),{'autoViz':True,'canAreaChart':True},'NLCD Land Cover')
Map.addLayer(nlcd.select(['landcover']).mode().set(nlcd.first().toDictionary()),{'autoViz':True,'canAreaChart':True},'NLCD Land Cover Mode')

# Use the shortened class names to clean up chart
Map.addLayer(nlcd.select(['landcover']).mode().set(nlcd_props_shortened_names),{'autoViz':True,'canAreaChart':True},'NLCD Land Cover Mode Shortened Names')





Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMS Land Cover
Adding layer: LCMS Land Cover Mode
Adding layer: NLCD Land Cover
Adding layer: NLCD Land Cover Mode
Adding layer: NLCD Land Cover Mode Shortened Names
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650316725&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-628d7ec95080.json


### Inline outputLib.charts equivalent

* For a single `ee.Image`, `outputLib.charts` produces a bar chart
* It auto-detects horizontal vs vertical orientation based on label length

In [19]:
# Inline: Bar chart for LCMS Land Cover mode image
lc_mode = lcms.select(['Land_Cover']).mode().set(lcms.first().toDictionary())
result = cl.summarize_and_chart(
    lc_mode,
    study_area,
    title='LCMS Land Cover Mode',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

,Trees,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask,Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix
0,60.82,18.44,0.0,8.97,0.03,1.53,0.0,0.76,0.49,0.31,1.93,6.54,0.16


In [20]:
# Inline: Bar chart for NLCD landcover mode with original (long) names
# Notice how the long class names make the chart hard to read — compare with the shortened version below
nlcd_mode_long = nlcd.select(['landcover']).mode().set(nlcd.first().toDictionary())
result = cl.summarize_and_chart(
    nlcd_mode_long,
    study_area,
    title='NLCD Land Cover Mode (Original Names)',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

,Open water: areas ... or soil.,Perennial ice/snow... cover.\r\n,"Developed, open sp...rposes.\r\n","Developed, low int... units.\r\n","Developed, medium ... units.\r\n",Developed high int... cover.\r\n,Barren land (rock/... cover.\r\n,Deciduous forest: ...change.\r\n,Evergreen forest: ...oliage.\r\n,Mixed forest: area... cover.\r\n,Shrub/scrub: areas...itions.\r\n,Grassland/herbaceo...razing.\r\n,Pasture/hay: areas...tation.\r\n,Cultivated crops: ...tilled.\r\n,Woody wetlands: ar... water.\r\n,Emergent herbaceou... water.\r\n
0,1.61,0.99,3.93,4.52,3.49,1.07,1.43,1.17,43.71,0.11,17.07,13.78,1.12,2.84,1.71,1.44


In [21]:
# Inline: Bar chart for NLCD landcover mode with shortened names
nlcd_mode = nlcd.select(['landcover']).mode().set(nlcd_props_shortened_names)
result = cl.summarize_and_chart(
    nlcd_mode,
    study_area,
    title='NLCD Land Cover Mode (Shortened Names)',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

,Open water,Perennial ice/snow,"Developed, open space","Developed, low intensity","Developed, medium intensity",Developed high intensity,Barren land (rock/sand/clay),Deciduous forest,Evergreen forest,Mixed forest,Shrub/scrub,Grassland/herbaceous,Pasture/hay,Cultivated crops,Woody wetlands,Emergent herbaceous wetlands
0,1.61,0.99,3.93,4.52,3.49,1.07,1.43,1.17,43.71,0.11,17.07,13.78,1.12,2.84,1.71,1.44


---
## Charting Images Without Color and Name Properties

* Sometimes you will have an image you add to the map with a min, max, and palette in the viz params that are thematic  or ordinal thematic data. If those images have no class_names, etc properties set, geeViz will try to render the chart properly using the given min, max, and palette if you specify `"areaChartParams":{"reducer":ee.Reducer.frequencyHistogram()}`

In [22]:
# Map-based approach
Map.clearMap()

def getMostRecentChange(c, code):
    def wrapper(img):
        yr = ee.Date(img.get("system:time_start")).get("year")
        return (
            ee.Image(yr)
            .int16()
            .rename(["year"])
            .updateMask(img.eq(code))
            .copyProperties(img, ["system:time_start"])
        )

    return c.map(wrapper)

mostRecentFastLossYear = getMostRecentChange(lcms.select(['Change']),3).max()

Map.addLayer(mostRecentFastLossYear, {'min':1985,'max':2023,'palette':["ffffe5", "fff7bc", "fee391", "fec44f", "fe9929", "ec7014", "cc4c02"],'canAreaChart':True,'areaChartParams':{'reducer':ee.Reducer.frequencyHistogram()}}, "Most Recent Fast Loss Year", True)

Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: Most Recent Fast Loss Year
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650320788&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-d592a536a941.json


### Inline outputLib.charts equivalent

* Pass `reducer=ee.Reducer.frequencyHistogram()` to force thematic treatment
* Without class properties, class values are used as labels

In [23]:
# Inline: Force frequencyHistogram for a non-class image
result = cl.summarize_and_chart(
    mostRecentFastLossYear,
    study_area,
    reducer=ee.Reducer.frequencyHistogram(),
    title='Most Recent Fast Loss Year',
    palette=["ffffe5", "fff7bc", "fee391", "fec44f", "fe9929", "ec7014", "cc4c02"],
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,0.02,0.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Charting Time Lapses 

* Time lapses also support area charting. All functionality is the same as the `Map.addLayer` methods.

* Band names can be specified in the `areaChartParams`. If they are not specified, but `bands` is specified in the visualization parameters, those bands will be used instead. Otherwise, all bands will be shown. 

In [24]:
# Map-based approach
Map.clearMap()

composites = ee.ImageCollection("projects/lcms-tcc-shared/assets/CONUS/Composites/Composite-Collection-yesL7")

years = list(range(1985,2024))

# Need to mosaic the tiled outputs for each year
composites = [composites.filter(ee.Filter.calendarRange(yr,yr,'year')).mosaic().set('system:time_start',ee.Date.fromYMD(yr,6,1).millis()) for yr in years]
composites = ee.ImageCollection(composites)

# Set up visualization parameters
viz = gil.vizParamsFalse10k
viz['canAreaChart']=True
viz['areaChartParams']={'bandNames':'blue,green,red,nir,swir1,swir2','palette':'00D,0D0,D00,D0D,0DD'}
Map.addTimeLapse(composites,gil.vizParamsFalse10k,'Composites')

Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: Composites
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650322344&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-90388ca96f34.json


### Inline outputLib.charts equivalent

* Use `band_names` to select specific bands for a continuous time series

In [25]:
# Inline: Composite time series of spectral bands
result = cl.summarize_and_chart(
    composites,
    study_area,
    band_names=['blue', 'green', 'red', 'nir', 'swir1', 'swir2'],
    title='Composite Band Means Over Time',
    palette=['00D', '0D0', 'D00', 'D0D', '0DD', 'DD0'],
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df.head())

,blue,green,red,nir,swir1,swir2
1985,985.580791,965.466387,937.616466,2133.543459,1873.828614,1221.201942
1986,1021.519651,1039.105031,997.072581,2133.979411,1693.730028,1102.372768
1987,925.358777,903.407423,860.822802,2018.785429,1743.441945,1127.207809
1988,1023.511417,994.061483,951.996446,2108.598152,1793.101677,1171.694232
1989,919.257158,888.920840,854.391704,1962.842510,1721.237147,1124.044893


---
## Charting Thematic Data without set properties

* You can chart thematic datasets that lack values, names, and palette properties by specifying the `ee.Reducer.frequencyHistogram()` as the reducer

* This is not the best method however. Charts will lack descriptive class names and colors

In [26]:
# Map-based approach: LCMAP example
Map.clearMap()


lcpri = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI").select(['b1'],['LC'])


Map.addTimeLapse(lcpri,{'min':1,'max':9,'canAreaChart':True,"areaChartParams":{'reducer':ee.Reducer.frequencyHistogram()}},'LCMAP LC Primary')

Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMAP LC Primary
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650324251&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-32990d5dcae3.json


### Inline outputLib.charts equivalent

In [27]:
# Inline: LCMAP without class properties — just uses class values as labels
result = cl.summarize_and_chart(
    lcpri.filterBounds(study_area),
    study_area,
    reducer=ee.Reducer.frequencyHistogram(),
    title='LCMAP LC Primary (no class properties)',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df.head())

,1,2,3,4,5,6,7,8
1985,8.34,8.19,19.67,50.31,1.56,2.73,4.29,4.90
1986,8.47,8.02,19.68,50.35,1.56,2.73,4.29,4.89
1987,8.56,7.87,19.71,50.39,1.56,2.73,4.29,4.89
1988,8.63,7.74,19.74,50.42,1.55,2.74,4.29,4.89
1989,8.68,7.67,19.73,50.44,1.55,2.74,4.29,4.89


---
### Setting properties for charting

* The easiest way to chart thematic data where each class has a number, name, and color, but lack the preset properties, is to set them on-the-fly

* These properties can then be used for charting and map rendering

In [28]:
# Map-based approach: LCMAP with manually set properties
Map.clearMap()

lcpri_palette = ['E60000','A87000','E3E3C2','1D6330','476BA1','BAD9EB','FFFFFF','B3B0A3','A201FF']
lc_names = ['Developed','Cropland','Grass/Shrub','Tree Cover','Water','Wetlands','Ice/Snow','Barren','Class Change']
lc_numbers = list(range(1,len(lcpri_palette)+1))

lcpri = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI").select(['b1'],['LC'])

lcpri = lcpri.map(lambda img:img.set({'LC_class_values':lc_numbers,'LC_class_names':lc_names,'LC_class_palette':lcpri_palette}))


Map.addTimeLapse(lcpri,{'autoViz':True,'canAreaChart':True,'areaChartParams':{'line':True,'sankey':True}},'LCMAP LC Primary')

Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMAP LC Primary
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650326811&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-6b9d313c64a8.json


### Inline outputLib.charts equivalent

In [29]:
# Inline: LCMAP with class properties — time series
result = cl.summarize_and_chart(
    lcpri,
    study_area,
    title='LCMAP LC Primary',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df.head())

,Developed,Cropland,Grass/Shrub,Tree Cover,Water,Wetlands,Ice/Snow,Barren
1985,8.34,8.19,19.67,50.31,1.56,2.73,4.29,4.90
1986,8.47,8.02,19.68,50.35,1.56,2.73,4.29,4.89
1987,8.56,7.87,19.71,50.39,1.56,2.73,4.29,4.89
1988,8.63,7.74,19.74,50.42,1.55,2.74,4.29,4.89
1989,8.68,7.67,19.73,50.44,1.55,2.74,4.29,4.89


In [30]:
# Inline: LCMAP sankey
result = cl.summarize_and_chart(
    lcpri,
    study_area,
    chart_type='sankey',
    transition_periods=[1985, 1990, 2000, 2005, 2018, 2021],
    title='LCMAP LC Primary Transitions',
    min_percentage=0.5,
)
sankey_df, sankey_fig, matrix_dict = result['df'], result['chart'], result['matrix']
display(HTML(sankey_fig))
print('Source-Target-Value table:')
display(sankey_df)
print('Transition matrices:')
for label, mdf in matrix_dict.items():
    print(f"\n{label}")
    display(mdf)

Source-Target-Value table:


,source,target,value,source_name,target_name,source_color,target_color,period
0,0,9,8.22,1985 Developed,1990 Developed,#E60000,#E60000,1985 -> 1990
1,1,10,7.56,1985 Cropland,1990 Cropland,#A87000,#A87000,1985 -> 1990
2,2,11,19.18,1985 Grass/Shrub,1990 Grass/Shrub,#E3E3C2,#E3E3C2,1985 -> 1990
3,3,12,50.14,1985 Tree Cover,1990 Tree Cover,#1D6330,#1D6330,1985 -> 1990
4,4,13,1.52,1985 Water,1990 Water,#476BA1,#476BA1,1985 -> 1990
5,5,14,2.73,1985 Wetlands,1990 Wetlands,#BAD9EB,#BAD9EB,1985 -> 1990
6,6,15,4.30,1985 Ice/Snow,1990 Ice/Snow,#FFFFFF,#FFFFFF,1985 -> 1990
7,7,16,4.89,1985 Barren,1990 Barren,#B3B0A3,#B3B0A3,1985 -> 1990
8,9,18,8.62,1990 Developed,2000 Developed,#E60000,#E60000,1990 -> 2000
9,10,18,0.56,1990 Cropland,2000 Developed,#A87000,#E60000,1990 -> 2000


Transition matrices:

1985 → 1990


,Developed,Cropland,Grass/Shrub,Tree Cover,Water,Wetlands,Ice/Snow,Barren,Class Change
Developed,8.22,0.03,0.07,0.00,0.01,0.00,0.0,0.00,0
Cropland,0.36,7.56,0.25,0.00,0.02,0.01,0.0,0.00,0
Grass/Shrub,0.13,0.04,19.18,0.31,0.00,0.00,0.0,0.00,0
Tree Cover,0.01,0.00,0.16,50.14,0.00,0.00,0.0,0.00,0
Water,0.01,0.01,0.01,0.00,1.52,0.01,0.0,0.00,0
Wetlands,0.00,0.00,0.00,0.00,0.00,2.73,0.0,0.00,0
Ice/Snow,0.00,0.00,0.00,0.00,0.00,0.00,4.3,0.00,0
Barren,0.00,0.00,0.00,0.00,0.00,0.00,0.0,4.89,0
Class Change,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0



1990 → 2000


,Developed,Cropland,Grass/Shrub,Tree Cover,Water,Wetlands,Ice/Snow,Barren,Class Change
Developed,8.62,0.04,0.05,0.00,0.01,0.00,0.0,0.00,0
Cropland,0.56,6.91,0.13,0.00,0.04,0.01,0.0,0.00,0
Grass/Shrub,0.68,0.19,18.24,0.53,0.02,0.00,0.0,0.00,0
Tree Cover,0.02,0.00,0.26,50.16,0.00,0.00,0.0,0.00,0
Water,0.01,0.01,0.00,0.00,1.53,0.01,0.0,0.00,0
Wetlands,0.00,0.00,0.00,0.00,0.01,2.73,0.0,0.00,0
Ice/Snow,0.00,0.00,0.00,0.00,0.00,0.00,4.3,0.00,0
Barren,0.00,0.00,0.00,0.00,0.00,0.00,0.0,4.89,0
Class Change,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0


... (2 outputs left out)


2005 → 2018


,Developed,Cropland,Grass/Shrub,Tree Cover,Water,Wetlands,Ice/Snow,Barren,Class Change
Developed,9.97,0.09,0.15,0.01,0.02,0.00,0.0,0.00,0
Cropland,0.40,6.22,0.30,0.00,0.04,0.01,0.0,0.00,0
Grass/Shrub,0.27,0.22,17.69,0.51,0.02,0.00,0.0,0.00,0
Tree Cover,0.02,0.00,1.29,49.21,0.00,0.01,0.0,0.01,0
Water,0.01,0.01,0.01,0.00,1.56,0.01,0.0,0.00,0
Wetlands,0.00,0.01,0.00,0.00,0.01,2.72,0.0,0.00,0
Ice/Snow,0.00,0.00,0.00,0.00,0.00,0.00,4.3,0.00,0
Barren,0.00,0.00,0.00,0.00,0.00,0.00,0.0,4.89,0
Class Change,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0



2018 → 2021


,Developed,Cropland,Grass/Shrub,Tree Cover,Water,Wetlands,Ice/Snow,Barren,Class Change
Developed,10.55,0.05,0.07,0.00,0.01,0.00,0.0,0.00,0
Cropland,0.07,6.37,0.10,0.01,0.01,0.00,0.0,0.00,0
Grass/Shrub,0.04,0.09,19.10,0.20,0.01,0.00,0.0,0.01,0
Tree Cover,0.00,0.00,1.18,48.17,0.05,0.00,0.0,0.32,0
Water,0.00,0.00,0.01,0.00,1.62,0.00,0.0,0.00,0
Wetlands,0.00,0.00,0.00,0.00,0.00,2.74,0.0,0.00,0
Ice/Snow,0.00,0.00,0.00,0.00,0.00,0.00,4.3,0.00,0
Barren,0.00,0.00,0.00,0.00,0.00,0.00,0.0,4.90,0
Class Change,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0


---
## Comparing map output versions

* One common task is to understand the differences between 2 or more model runs. This can be challenging.

* This example will show three versions of LCMS products and how they relate

* This approach makes comparing the maps and their respective class counts relatively easy

* This idea could be adapted to comparing different thematic or continuous outputs

In [31]:
# Map-based approach
Map.clearMap()

# Bring in 3 different LCMS versions
lcms_2020 = ee.ImageCollection("USFS/GTAC/LCMS/v2020-5").filter('study_area=="CONUS"')
lcms_2021 = ee.ImageCollection("USFS/GTAC/LCMS/v2021-7").filter('study_area=="CONUS"')
lcms_2022 = ee.ImageCollection("USFS/GTAC/LCMS/v2022-8").filter('study_area=="CONUS"')
lcms_2025 = ee.ImageCollection("projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11").filter('study_area=="CONUS"')

# Choose a year to compare (any year 1985-2020)
year = 2010

# Filter off the image and set that image to the version year
lcms_2020 = lcms_2020.filter(ee.Filter.calendarRange(year,year,'year')).first().set({'year':2020,'system:time_start':ee.Date.fromYMD(2020,6,1).millis()})
lcms_2021 = lcms_2021.filter(ee.Filter.calendarRange(year,year,'year')).first().set({'year':2021,'system:time_start':ee.Date.fromYMD(2021,6,1).millis()})
lcms_2022 = lcms_2022.filter(ee.Filter.calendarRange(year,year,'year')).first().set({'year':2022,'system:time_start':ee.Date.fromYMD(2022,6,1).millis()})
lcms_2025 = lcms_2025.filter(ee.Filter.calendarRange(year,year,'year')).first().set({'year':2025,'system:time_start':ee.Date.fromYMD(2025,6,1).millis()})

# Construct the image collection
c = ee.ImageCollection([lcms_2020,lcms_2021,lcms_2022,lcms_2025])

# Add the collection as a timelapse
# Will need to specify the transition years as the years used for each image, otherwise geeViz will default to only showing the first and last year
# Note that if you specify the sankey transition periods, any periods you enter in the geeViz UI will not be used for that layer
for bn in ['Change','Land_Cover','Land_Use']:
  Map.addTimeLapse(c.select([bn]),{'autoViz':True,'canAreaChart':True,'areaChartParams':{'sankey':True,
                                                                            'sankeyTransitionPeriods':[[2020,2020],[2021,2021],[2022,2022],[2025,2025]]
                                                                            }}, f"LCMS {bn.replace('_',' ')} Comparison {year}")

Map.turnOnAutoAreaCharting()
Map.view()

Adding layer: LCMS Change Comparison 2010
Adding layer: LCMS Land Cover Comparison 2010
Adding layer: LCMS Land Use Comparison 2010
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650329570&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-89a58da625a1.json


C:\RCR\geeVizBuilder\venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for USFS/GTAC/LCMS/v2020-5! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2020-5

  warnings.warn(warning, category=DeprecationWarning)
C:\RCR\geeVizBuilder\venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for USFS/GTAC/LCMS/v2021-7! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2021-7

  warnings.warn(warning, category=DeprecationWarning)
C:\RCR\geeVizBuilder\venv\Lib\site-packages\ee\d

### Inline outputLib.charts equivalent

In [32]:
# Inline: Version comparison — time series + sankey
for bn in ['Change', 'Land_Cover', 'Land_Use']:
    # Time series
    result = cl.summarize_and_chart(
        c.select([bn]),
        study_area,
        title=f'LCMS {bn.replace("_", " ")} Comparison {year}',
    )
    df, fig = result['df'], result['chart']
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

    # Sankey
    result = cl.summarize_and_chart(
        c.select([bn]),
        study_area,
        chart_type='sankey',
        # These must be the years the comparison collection above
        # actually carries -- 2020/2021/2022/2025, one per LCMS
        # version. Asking for 2023 (a version that no longer leads)
        # leaves the last transition with no 'to' image, and Earth
        # Engine reports it as
        #   Image.select: Band pattern 'to' did not match any bands.
        #   Available bands: [from]
        # which names the symptom and not the cause.
        transition_periods=[2020, 2021, 2022, 2025],
        title=f'LCMS {bn.replace("_", " ")} Version Transitions {year}',
        min_percentage=0.5,
    )
    sankey_df, sankey_fig, matrix_dict = result['df'], result['chart'], result['matrix']
    display(HTML(sankey_fig))
    print(f'Transition matrices for {bn}:')
    for label, mdf in matrix_dict.items():
        print(f"\n{label}")
        display(mdf)

Transition matrices for Change:

2020 → 2021


,Stable,Slow Loss,Fast Loss,Gain,Non-Processing Area Mask
Stable,86.45,2.54,0.02,0.71,0.0
Slow Loss,2.76,5.10,0.02,0.02,0.0
Fast Loss,0.04,0.03,0.17,0.00,0.0
Gain,0.94,0.01,0.00,0.97,0.0
Non-Processing Area Mask,0.22,0.00,0.00,0.00,0.0



2021 → 2022


,Stable,Slow Loss,Fast Loss,Gain,Non-Processing Area Mask
Stable,85.68,3.48,0.06,1.19,0.0
Slow Loss,3.05,4.53,0.04,0.07,0.0
Fast Loss,0.03,0.02,0.15,0.00,0.0
Gain,0.81,0.03,0.00,0.86,0.0
Non-Processing Area Mask,0.01,0.00,0.00,0.00,0.0



2022 → 2025


,Stable,Slow Loss,Fast Loss,Gain,Non-Processing Area Mask
Stable,0,0,0.01,0.02,0.03
Slow Loss,0,0,0.00,0.00,0.00
Fast Loss,0,0,0.00,0.00,0.00
Gain,0,0,0.00,0.00,0.00
Non-Processing Area Mask,0,0,0.00,0.00,0.00


... (8 outputs left out)


2021 → 2022


,Agriculture,Developed,Forest,Non-Forest Wetland,Other,Rangeland or Pasture,Non-Processing Area Mask
Agriculture,3.01,0.12,0.10,0.02,0.00,0.49,0.0
Developed,0.18,5.66,0.22,0.00,0.07,0.29,0.0
Forest,0.07,0.13,60.08,0.09,0.34,2.49,0.0
Non-Forest Wetland,0.02,0.00,0.16,0.26,0.02,0.06,0.0
Other,0.00,0.02,0.19,0.01,4.62,0.31,0.0
Rangeland or Pasture,1.08,0.51,2.70,0.07,0.83,15.77,0.0
Non-Processing Area Mask,0.00,0.00,0.00,0.00,0.01,0.00,0.0



2022 → 2025


,Agriculture,Developed,Forest,Non-Forest Wetland,Other,Rangeland or Pasture,Non-Processing Area Mask
Agriculture,2.88,0.57,0.03,0.00,0.86,0.0,0
Developed,0.05,6.25,0.03,0.00,0.10,0.0,0
Forest,0.09,1.30,57.93,0.16,3.97,0.0,0
Non-Forest Wetland,0.04,0.01,0.03,0.02,0.35,0.0,0
Other,0.01,0.23,0.06,4.83,0.77,0.0,0
Rangeland or Pasture,1.28,1.37,0.44,0.11,16.22,0.0,0
Non-Processing Area Mask,0.00,0.00,0.00,0.00,0.00,0.0,0


---
## Other charting summary zone selection methods

* All examples have simply used the map extent as the zone to chart

* There are other methods available

* This example will show how to add a featureCollection to interactively select areas to summarize

* All area selection will happen in the geeViz UI, under `Tools -> Area Tools -> Select an Area on Map`

* This example also demonstrates the use of the stacked bar `chartType` and turning off the first and last MTBS burn severity classes in the chart

In [33]:
# Map-based approach
Map.clearMap()


for bn in ['Change','Land_Cover','Land_Use']:
  Map.addLayer(lcms.select([bn]),{'autoViz':True,'canAreaChart':True}, f"LCMS {bn.replace('_',' ')}  ")


# Bring in MTBS burn boundaries
mtbsBoundaries = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
mtbsBoundaries = mtbsBoundaries.map(
    lambda f: f.set("system:time_start", f.get("Ig_Date"))
)
# Bring in MTBS burn severity mosaics
mtbsSeverity = ee.ImageCollection("USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1").filter(ee.Filter.stringContains('system:index','_CONUS_')).select([0],['Severity'])
Map.addLayer(mtbsSeverity,{'autoViz':True,'canAreaChart':True,'areaChartParams':{'chartType':'stacked-bar',"visible": [False, True, True, True, True, True, False],}}, "MTBS Burn Severity")

# For area charting you can select areas to chart a number of ways. One method is using a map layer that is selectable by clicking on each feature.
Map.addSelectLayer(
    mtbsBoundaries,
    {
        "strokeColor": "00F",
        "selectLayerNameProperty": "Incid_Name",
    },
    "MTBS Fire Boundaries"
)

Map.turnOnSelectionAreaCharting()
Map.view()

Adding layer: LCMS Change  
Adding layer: LCMS Land Cover  
Adding layer: LCMS Land Use  
Adding layer: MTBS Burn Severity
Adding layer: MTBS Fire Boundaries
Starting webmap
Using eeCreds proxy at http://127.0.0.1:1112/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1112/geoview/geoview.html?v=1795650338200&doc=../geeView/src/gee/gee-run/map-1c941fba55fb7648-14332ac47bff.json


### Inline outputLib.charts equivalent

* With `outputLib.charts` you can pass any `ee.FeatureCollection` as the geometry
* For a single-feature FC (e.g. a state boundary), it behaves like a single geometry
* This is useful for scripted, reproducible analyses where you want to specify the exact zone
* Below we chart all LCMS thematic bands and MTBS Burn Severity over Colorado

In [34]:
# Inline: LCMS thematic bands summarized over Colorado
co = ee.FeatureCollection('TIGER/2018/States').filter(ee.Filter.eq('NAME', 'Colorado'))

for bn in ['Change', 'Land_Cover', 'Land_Use']:
    result = cl.summarize_and_chart(
        lcms.select([bn]),
        co,
        title=f'LCMS {bn.replace("_", " ")} \u2014 Colorado',
    )
    df, fig = result['df'], result['chart']
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [35]:
# Inline: MTBS Burn Severity stacked bar chart over Colorado
mtbsSeverity = ee.ImageCollection("USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1").filter(ee.Filter.stringContains('system:index', '_CONUS_')).select([0],['Severity'])

result = cl.summarize_and_chart(
    mtbsSeverity,
    co,
    title='MTBS Burn Severity \u2014 Colorado',
    chart_type='bar',
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df.head())

,Background,Unburned to Low,Low,Moderate,High,Increased Greenness,Non-Mapping Area
1984,99.99,0.00,0.01,0.00,0.0,0.0,0.0
1985,100.00,0.00,0.00,0.00,0.0,0.0,0.0
1986,99.93,0.00,0.04,0.02,0.0,0.0,0.0
1987,99.97,0.01,0.01,0.01,0.0,0.0,0.0
1988,99.93,0.01,0.03,0.02,0.0,0.0,0.0


---
## Multi-Feature `reduceRegions` (outputLib.charts only)

* When you pass a multi-feature `ee.FeatureCollection` as the geometry **and** set `feature_label` to the name of a feature property, `outputLib.charts` uses `reduceRegions` to compute per-feature statistics in a single server call
* The result is a grouped bar chart where each group is a feature (fire, state, watershed, etc.) and each bar within the group is a class
* This is far more efficient than looping over features individually

* In this example we find the 5 largest MTBS fires and summarize both LCMS Land Cover and Annual NLCD over each fire perimeter

In [36]:
# Get the 5 largest MTBS fire perimeters by burned area (BurnBndAc)
mtbs_perims = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
top5_fires = mtbs_perims.filter(ee.Filter.notNull(["BurnBndAc", "Ig_Year", "Incid_Name"])).sort("BurnBndAc", False).limit(5)

# Shorten fire names for readability
top5_fires = top5_fires.map(
    lambda f: f.set("Fire", ee.String(f.get("Incid_Name")).cat(" (").cat(ee.Number(f.get("Ig_Year")).format("%d")).cat(")")))

print("5 largest MTBS fires:")
for feat in top5_fires.getInfo()["features"]:
    p = feat["properties"]
    fire_name = p.get("Fire", p.get("Incid_Name"))
    acres = p["BurnBndAc"]
    print(f"  {fire_name}: {acres:,.0f} acres")

5 largest MTBS fires:


In [37]:
# LCMS Land Cover mode over the 5 largest fires (reduceRegions)
lc_mode = lcms.select(["Land_Cover"]).mode().set(lcms.first().toDictionary())

result = cl.summarize_and_chart(
    lc_mode,
    top5_fires,
    feature_label="Fire",
    title="LCMS Land Cover Mode \u2014 5 Largest MTBS Fires",
    width=800,
    height=450,
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

""
0


In [38]:
# Annual NLCD landcover mode over the 5 largest fires (reduceRegions)
nlcd_mode_fires = nlcd.select(["landcover"]).mode().set(nlcd_props_shortened_names)

result = cl.summarize_and_chart(
    nlcd_mode_fires,
    top5_fires,
    feature_label="Fire",
    title="NLCD Land Cover Mode \u2014 5 Largest MTBS Fires",
    width=800,
    height=450,
)
df, fig = result['df'], result['chart']
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
display(df)

""
0


---
## Area Format Options (outputLib.charts only)

* By default, thematic charts show percentages
* You can switch to hectares, acres, or raw pixel counts using the `area_format` parameter

In [39]:
# Inline: Same data in hectares
result = cl.summarize_and_chart(
    lcms.select(['Land_Cover']),
    study_area,
    area_format='Hectares',
    title='LCMS Land Cover (Hectares)'
)
df_ha, fig_ha = result['df'], result['chart']
display(HTML(fig_ha.to_html(full_html=False, include_plotlyjs="cdn")))

# And in acres
result = cl.summarize_and_chart(
    lcms.select(['Land_Cover']),
    study_area,
    area_format='Acres',
    title='LCMS Land Cover (Acres)'
)
df_ac, fig_ac = result['df'], result['chart']
display(HTML(fig_ac.to_html(full_html=False, include_plotlyjs="cdn")))

---
## Using the Lower-Level Functions (outputLib.charts only)

* `summarize_and_chart` is a convenience wrapper
* You can also call the lower-level functions directly for more control:
  - `cl.get_obj_info()` — inspect the object type and class metadata
  - `cl.zonal_stats()` — get just the DataFrame (no chart)
  - `cl.chart_time_series()` / `cl.chart_bar()` / `cl.chart_sankey()` — build charts from DataFrames

In [40]:
# Step-by-step: inspect, compute stats, then chart
lc = lcms.select(['Land_Cover'])

# 1. Inspect the object
info = cl.get_obj_info(lc)
print('Object info:')
for k, v in info.items():
    if k != 'class_info':
        print(f'  {k}: {v}')
print(f'  class_info bands: {list(info["class_info"].keys())}')

# 2. Compute zonal stats (returns just the DataFrame)
df = cl.zonal_stats(lc, study_area, area_format='Percentage')
print(f'\nDataFrame shape: {df.shape}')
display(df.head())

# 3. Build chart from the DataFrame
colors = info['class_info']['Land_Cover']['class_palette']
fig = cl.chart_time_series(
    df,
    colors=colors,
    title='LCMS Land Cover (step-by-step)',
    y_label='% Area'
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

Object info:
  obj_type: ImageCollection
  band_names: ['Land_Cover']
  is_thematic: True
  size: 164
  class_info bands: ['Land_Cover']



DataFrame shape: (41, 13)


,Trees,Grass/Forb/Herb,Barren & Grass/Forb/Herb Mix,Barren or Impervious,Snow or Ice,Water,Non-Processing Area Mask,Shrubs & Trees Mix,Grass/Forb/Herb & Trees Mix,Barren & Trees Mix,Shrubs,Grass/Forb/Herb & Shrubs Mix,Barren & Shrubs Mix
1985,56.41,21.09,0.01,6.71,0.10,1.43,4.21,0.80,0.51,0.27,3.90,4.43,0.14
1986,58.33,20.24,0.01,8.62,0.10,1.46,0.90,0.86,0.54,0.31,3.10,5.39,0.16
1987,58.78,20.37,0.01,8.69,0.06,1.45,0.08,0.88,0.54,0.34,2.91,5.73,0.16
1988,59.26,20.07,0.01,8.41,0.06,1.44,0.03,0.88,0.57,0.32,2.84,5.94,0.17
1989,59.78,19.61,0.01,8.31,0.05,1.41,0.01,0.85,0.57,0.34,2.89,6.01,0.16
